# Tablas de caracteristicas

In [0]:
import mlflow

catalog = "main"
schema = "dbdemos_fs_travel"

# Experimento y modelo
xp_name = f"compra_viaje_experimento_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S')}"
xp_path = "/Workspace/Proyectos_Dev/databricks-medallion/Experiments"
model_name = "purchase_travel_model"
model_full_name = f"{catalog}.{schema}.{model_name}"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"""USE `{catalog}`.`{schema}`""")    

In [0]:
%sql 
SELECT * FROM user_features_advanced LIMIT 5;
SELECT * FROM destination_features_advanced LIMIT 5;

In [0]:
df = spark.table('travel_purchase').select('user_id', 'destination_id', 'purchased', 'ts').sample(fraction=0.3, seed=42)

# Split based on time to define a training and inference set (we'll do train+eval on the past & test in the most current value)
training_labels_df = df.where("ts < '2022-11-01'")
test_labels_df = df.where("ts >= '2022-11-01'")

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from databricks.feature_store import feature_table, FeatureLookup

fe_table_name_users = f"{catalog}.{schema}.user_features_advanced"
fe_table_name_destinations = f"{catalog}.{schema}.destination_features_advanced"

fe = FeatureEngineeringClient()

model_feature_lookups = [
      FeatureLookup(
          table_name=fe_table_name_destinations,
          lookup_key="destination_id",
          timestamp_lookup_key="ts"
      ),
      FeatureLookup(
          table_name=fe_table_name_users,
          lookup_key="user_id",
          feature_names=["mean_price_7d", "last_6m_purchases", "tenure_days", "age", "gender", "income_bracket", "loyalty_tier", "billing_state"], 
          timestamp_lookup_key="ts"
      )
]

# fe.create_training_set will look up features in model_feature_lookups with matched key from training_labels_df
training_set = fe.create_training_set(
    df=training_labels_df, # joining the original Dataset, with our FeatureLookupTable
    feature_lookups=model_feature_lookups,
    exclude_columns=["ts", "destination_id", "user_id"], # exclude id columns as we don't want them as feature
    label='purchased',
)

test_set = fe.create_training_set(
    df=test_labels_df,
    feature_lookups=model_feature_lookups,
    exclude_columns=["ts", "destination_id", "user_id"],
    label="purchased",  # keep label here for offline evaluation
)

training_pd = training_set.load_df().toPandas()
test_pd = test_set.load_df().toPandas()
X_train, y_train = training_pd.drop("purchased", axis=1), training_pd["purchased"]
X_test,  y_test  = test_pd.drop("purchased", axis=1),  test_pd["purchased"]

In [0]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify feature types
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numerical features: {len(numerical_features)}, Categorical: {len(categorical_features)}")

numeric_transformer = Pipeline([("scaler", StandardScaler())])
categorical_transformer = Pipeline(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
)
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier

best_model = None
best_auc = -1
best_name = None

# Candidate models
candidates = {
    "log_reg": LogisticRegression(max_iter=500),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "gbm": GradientBoostingClassifier(random_state=42),
    "lightgbm": LGBMClassifier(random_state=42)
}

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.metrics import roc_auc_score, accuracy_score

print(f"Starting experiment: {xp_name}")

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(xp_path)
x_sample = X_train.sample(1, random_state=42)
y_sample = y_train.sample(1, random_state=42)
signature = infer_signature(X_train, y_sample)

for name, clf in candidates.items():
    with mlflow.start_run(run_name=name):
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])
        model.fit(X_train, y_train)

        y_pred_proba = model.predict_proba(X_test)[:, 1]
        auc = sklearn.metrics.roc_auc_score(y_test, y_pred_proba)
        acc = sklearn.metrics.accuracy_score(y_test, (y_pred_proba > 0.5).astype(int))

        print(f"{name:15s} | AUC: {auc:.4f} | ACC: {acc:.4f}")

        #  Log parameters and metrics
        mlflow.log_param("model_name", name)
        mlflow.log_metrics({"auc": auc, "accuracy": acc})

        #  Log model for each candidate
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)

        if auc > best_auc:
            best_auc, best_model, best_name = auc, model, name
            mlflow.set_tag("best_model", "true")

print(f"\nBest model: {best_name} (AUC={best_auc:.4f})")

## Guardar mejor modelo en MLflow registry

In [0]:
# Conda environment cleanup
env = mlflow.pyfunc.get_default_conda_env()
for dep in env["dependencies"]:
    if isinstance(dep, dict) and "pip" in dep:
        dep["pip"] = [pkg for pkg in dep["pip"] if not pkg.startswith("mlflow==")]
        dep["pip"].insert(0, f"mlflow=={mlflow.__version__}")
        dep["pip"].insert(1, "numpy<2.0")

x_sample = X_train.sample(1, random_state=42)
signature = infer_signature(X_train, best_model.predict(X_train))

#All in ONE MLflow run — preserves feature lineage
with mlflow.start_run(run_name=f"{xp_name}_{best_name}") as run:
    mlflow.log_metric("test_auc", best_auc)
    mlflow.log_param("best_model_type", best_name)
    mlflow.log_input(mlflow.data.from_pandas(X_train.sample(10, random_state=42)), "training_sample")

    fe.log_model(
        model=best_model,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,     # FS lineage preserved here       
        input_example=x_sample,
        signature=signature,
        registered_model_name=model_full_name,
        conda_env=env
    )

print(f"\n Model logged and registered with FS metadata: {model_full_name}")
print("This model will now support automatic feature lookup in Model Serving.")

In [0]:
from mlflow.tracking import MlflowClient

def get_last_model_version(model_name):
    client = MlflowClient(registry_uri="databricks-uc")
    versions = client.search_model_versions(f"name='{model_name}'")
    latest = max(versions, key=lambda v: int(v.version))
    return latest

latest_model = get_last_model_version(model_full_name)
model_alias = "production"
if len(latest_model.aliases) == 0 or latest_model.aliases[0] != model_alias:
    print(f"updating model version {latest_model.version} to alias {model_alias}")
    mlflow_client = MlflowClient(registry_uri="databricks-uc")
    mlflow_client.set_registered_model_alias(model_full_name, model_alias, version=latest_model.version)

## Servicio endpoint del modelo

In [0]:
# Get latest version dynamically from the registry
client = mlflow.deployments.get_deploy_client("databricks")
latest_version = latest_model.version
ms_endpoint_name = f"{catalog}_{schema}_ms-travel-recommendation"[:50]

endpoint_config = {
    "served_entities": [
        {
            "entity_name": model_full_name,       # full UC model path
            "entity_version": str(latest_version),
            "workload_size": "Small",             # Small / Medium / Large
            "scale_to_zero_enabled": True
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": f"{model_name}-{latest_version}",
                "traffic_percentage": 100
            }
        ]
    }
}
# For first time creation
endpoint = client.create_endpoint(name=ms_endpoint_name,config=endpoint_config)
# For updating the endpoint
#endpoint = client.update_endpoint(ms_endpoint_name, endpoint_config)

print(f"Endpoint '{ms_endpoint_name}' created or updated successfully.")
print(f"Model served: {model_full_name} (version {latest_version})")

In [0]:
if wait_until_endpoint_ready(ms_endpoint_name, timeout=900, sleep_time=30):
    print("Endpoint confirmed ready — safe to send inference or feature requests.")
else:
    raise TimeoutError(f"Endpoint '{ms_ndpoint_name}' is not ready yet.")